# Kevin 24-Well Biologic Workflow

Single notebook for the plan-driven 24-well run: load the experiment plan, fill reactor wells, run the Biologic batch, inspect results, optionally ultrasonic-clean the electrode, then flush wells. Run Kevin's setup/connect cells first in the same kernel so `otflex` and MQTT clients are available.

Edit [kevin_24well_biologic_plan.json](../../data/experiment_plans/kevin_24well_biologic_plan.json) to change well recipes, source wells, and Biologic techniques. Helper code lives in [kevin_24well_batch.py](../../src/workflows/kevin_24well_batch.py).

## 1. Load Plan And Shared Controls

This section finds the repository root, imports the shared helpers, loads the JSON plan, and decides whether this run should be hardware-active or dry-run based on whether setup/connect has already been run.

In [ ]:
import json
import sys
from pathlib import Path

# Locate repo root even if this notebook is launched from notebooks/workflows.
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src").exists():
    repo_root = repo_root.parent
if not (repo_root / "src").exists():
    raise RuntimeError("Could not find repository root containing src/")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.workflows.kevin_24well_batch import (
    BiologicBatchRunner,
    auto_dry_run,
    enabled_experiments,
    experiments_dataframe,
    fill_reactor_from_plan,
    flush_well_params,
    flush_wells_from_plan,
    load_plan,
    move_electrode_to_well,
    pick_electrode_tool,
    require_setup_namespace,
    return_electrode_tool,
    solution_transfer_groups,
    ultrasonic_clean_electrode,
    write_batch_summary,
)

PLAN_PATH = repo_root / "data" / "experiment_plans" / "kevin_24well_biologic_plan.json"
DRY_RUN = None  # None = auto: dry-run unless setup/connect is ready. Set True/False to override.
RESUME_FROM_WELL = None  # Example: "C3" to skip earlier enabled wells.
STOP_AFTER_WELL = None   # Example: "D3" for a partial run.

DRY_RUN, setup_missing = auto_dry_run(globals(), DRY_RUN)
plan = load_plan(PLAN_PATH)
batch_out_dir = repo_root / "data" / "out" / plan["batch_id"]
local_batch_dir = batch_out_dir / "biologic"
batch_out_dir.mkdir(parents=True, exist_ok=True)
local_batch_dir.mkdir(parents=True, exist_ok=True)

print(f"Loaded plan: {PLAN_PATH}")
print(f"Batch ID: {plan['batch_id']}")
print(f"DRY_RUN: {DRY_RUN}")
if setup_missing:
    print("Setup not ready:", ", ".join(setup_missing))
experiments_dataframe(plan)

## 2. Review Planned Work

This section previews the liquid-transfer groups and the enabled Biologic experiments before any hardware action. It is safe to run repeatedly.

In [ ]:
transfer_groups = solution_transfer_groups(plan)
print("Solution transfer groups:")
for idx, group in enumerate(transfer_groups, start=1):
    print(
        f"{idx:02d}. {group['liquid']} from {group['source_labware']}.{group['source_well']} "
        f"-> {len(group['target_wells'])} wells at {group['volume_uL']} uL each"
    )
    print("    wells:", ", ".join(group["target_wells"]))

experiments = enabled_experiments(plan)
if RESUME_FROM_WELL:
    start_idx = next(i for i, exp in enumerate(experiments) if exp["well"] == RESUME_FROM_WELL)
    experiments = experiments[start_idx:]
if STOP_AFTER_WELL:
    stop_idx = next(i for i, exp in enumerate(experiments) if exp["well"] == STOP_AFTER_WELL)
    experiments = experiments[:stop_idx + 1]

print(f"\nQueued {len(experiments)} enabled Biologic experiments:")
for exp in experiments:
    experiment_id = exp.get("biologic", {}).get("experiment_id", plan["defaults"]["biologic"].get("experiment_id"))
    print(exp["well"], experiment_id)

## 3. Fill Reactor Wells

This section executes the plan-driven liquid transfers into the 24-well reactor. Skip it if all reactor wells are already filled with the intended solutions.

In [ ]:
if not DRY_RUN:
    require_setup_namespace(globals(), ["otflex"])

otflex_handle = globals().get("otflex")
transfers = await fill_reactor_from_plan(otflex_handle, plan, dry_run=DRY_RUN)

with open(batch_out_dir / "solution_filling_transfers.json", "w", encoding="utf-8") as f:
    json.dump(transfers, f, indent=2)

print(f"Recorded transfer plan to: {batch_out_dir / 'solution_filling_transfers.json'}")
print("DRY_RUN was", DRY_RUN)

## 4. Configure Biologic Remote Run

This section defines the SSH connection to the remote Biologic host and applies any resume/stop limits from the shared controls above.

In [ ]:
biologic_params = {
    "host": "192.168.0.102",
    "device_address": "USB0",
    "username": "sdl1_",
    "ssh_port": 22,
    "key_file": None,
    "channel": 4,
    "conda_env": "ot2_workflow",
    "python_executable": "python",
    "remote_work_dir": "AC_OTflex_remote",
    "remote_results_dir": None,
    "ocv_check_duration_s": 2,
    "host_start_timeout_s": 45,
}

print(f"Remote Biologic host: {biologic_params['username']}@{biologic_params['host']}:{biologic_params['ssh_port']}")
print(f"Queued wells: {', '.join(exp['well'] for exp in experiments)}")

## 5. Run Biologic Batch

This section picks up the electrode tool, moves through each queued reactor well, runs the remote Biologic experiment, downloads matching result files, and returns the tool at the end.

In [ ]:
if not DRY_RUN:
    require_setup_namespace(globals(), ["otflex"])
if "ec_state" not in globals():
    ec_state = {"tool_attached": False, "current_well": None, "at_ultrasonic": False}

results = []
if DRY_RUN:
    print("DRY RUN: no robot movement and no Biologic SSH commands will be executed.")
    for exp in experiments:
        print(f"[DRY RUN] move electrode to {exp['well']} and run Biologic experiment")
else:
    runner = BiologicBatchRunner(biologic_params)
    runner.connect()
    try:
        if not ec_state.get("tool_attached", False):
            await pick_electrode_tool(otflex, ec_state, plan["defaults"])

        for idx, exp in enumerate(experiments, start=1):
            well = exp["well"]
            print("=" * 70)
            print(f"[{idx}/{len(experiments)}] Well {well}")
            await move_electrode_to_well(otflex, ec_state, well, plan["defaults"])
            result = runner.run_experiment(plan, exp, local_batch_dir)
            results.append(result)
            write_batch_summary(results, local_batch_dir)
            print(f"Completed {well}: {result['downloaded_files']}")
    finally:
        try:
            await return_electrode_tool(otflex, ec_state, plan["defaults"])
        finally:
            runner.close()

summary = write_batch_summary(results, local_batch_dir) if results else None
summary if summary is not None else "No new results from this run."

## 6. Display Experiment Results

This section reloads the latest batch summary from disk so you can inspect downloaded files and per-well status before cleaning or flushing.

In [ ]:
import pandas as pd

summary_path = local_batch_dir / "batch_summary.csv"
if summary_path.exists():
    summary_df = pd.read_csv(summary_path)
    display(summary_df)
else:
    print(f"No summary file yet: {summary_path}")

## 7. Ultrasonic Clean Electrode

This section picks up the electrode tool if needed, moves it through ultrasonic stations A and B, runs the ultrasonic transducer at each station, and returns the tool. It is placed after result visualization and before well flushing.

In [ ]:
ULTRASONIC_STATIONS = ("A", "B")
ULTRASONIC_CHANNEL = 1
ULTRASONIC_DURATION_S = 15.0

if "ec_state" not in globals():
    ec_state = {"tool_attached": False, "current_well": None, "at_ultrasonic": False}

ultra_client = globals().get("ultra")
if ultra_client is None:
    rt = getattr(getattr(globals().get("otflex"), "mod", None), "_RT", None)
    ultra_client = getattr(rt, "mqtt_ultra", None) if rt is not None else None

if not DRY_RUN:
    require_setup_namespace(globals(), ["otflex"])

await ultrasonic_clean_electrode(
    globals().get("otflex"),
    ultra_client,
    ec_state,
    plan["defaults"],
    dry_run=DRY_RUN,
    stations=ULTRASONIC_STATIONS,
    channel=ULTRASONIC_CHANNEL,
    duration_s=ULTRASONIC_DURATION_S,
)
print("Ultrasonic cleaning section complete. DRY_RUN was", DRY_RUN)

## 8. Flush Wells

This section flushes the enabled reactor wells using the OTFlex `flushWell` runtime action. The parameter block is generated from the same plan so labware names stay aligned.

In [ ]:
FLUSH_WELLS = [exp["well"] for exp in experiments]
FLUSH_OVERRIDES = {
    "source_well": "C1",
    "to_offset_z": 5.0,
    "in_pump_id": 2,
    "out_pump_id": 3,
    "in_time_ms": 2000,
    "out_time_ms": 5000,
    "repeats": 4,
    "purge_ms": 0,
    "return_dz": 12.0,
}

if "ec_state" not in globals():
    ec_state = {"tool_attached": False, "current_well": None, "at_ultrasonic": False}

flush_params = flush_well_params(plan, wells=FLUSH_WELLS, **FLUSH_OVERRIDES)
if DRY_RUN:
    print("DRY RUN: no flush movement or pump commands will be executed.")
    print(json.dumps(flush_params, indent=2))
else:
    require_setup_namespace(globals(), ["otflex"])
    if ec_state.get("tool_attached", False):
        print("Returning currently attached electrode tool before flushing.")
        await return_electrode_tool(otflex, ec_state, plan["defaults"])
    flush_params = await flush_wells_from_plan(
        otflex,
        plan,
        dry_run=False,
        wells=FLUSH_WELLS,
        **FLUSH_OVERRIDES,
    )
    ec_state.update({"tool_attached": False, "current_well": None, "at_ultrasonic": False})

print(
    f"Flush section complete for {len(FLUSH_WELLS)} wells "
    f"(OUT -> (IN/OUT)x{flush_params['repeats']} -> OUT)."
)